In [ ]:
import sys
import os
import glob 
import time
os.environ['CUDA_VISIBLE_DEVICES']=""
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
import numpy as np
import dask.dataframe as dd
from matplotlib import pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.stats import boxcox
from scipy import stats
import polars as pl
import pandas as pd
import re
from itertools import groupby
from datetime import datetime, timedelta
import pyarrow.dataset as ds
from pathlib import Path

In [ ]:
np.set_printoptions(suppress=True)
pd.options.display.float_format = '{:.3f}'.format

In [ ]:
EMB_PATH = Path('../../../data/embeddings/')

# AGE

In [ ]:
def log_log_transform(y, epsilon=1e-10):
    """Apply log(log(y)) transformation for doubly exponential data"""
    # Ensure y > 0 for double log
    y_positive = y - y.min() + 1  # Shift to ensure y > 0
    return np.log(np.log(y_positive + epsilon))

In [ ]:
def age_anomaly_target(df):
    df = df.copy()

    # считаем mean, std для cv 
    df_mean = df['post_amount'].apply(lambda x: np.mean(x) if len(x) > 0 else np.nan)
    df_std = df['post_amount'].apply(lambda x: np.std(x) if len(x) > 0 else np.nan)

    # cчитаем CV
    df["cv"] = df_std / df_mean

    # yаходим пороги
    q95_cv = df["cv"].quantile(0.95)

    # если аномалия чисто по cv плохая можно попробовать дополнительно к cv взять q95 или q05
    #q05_mean = df_mean.quantile(0.05) # Добавим условие низкого чека, как мы обсуждали

    # cоздаем таргет 
    df["target_anomaly"] = (
        (df["cv"] > q95_cv)
    ).astype(int)

    return df["target_anomaly"]


In [ ]:
def get_regression_target(interval_feature: pd.Series, target_feature: pd.Series, horizon: int = 30):
    return [
        np.log1p(
            np.sum(np.asarray(a)[(d - d[0]) < horizon])
        ) for a, d in zip(
            target_feature, 
            interval_feature
        )
    ]

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_age_coles_ntp_TILL.parquet'
embeddings_path_prepocessed = '../../../data/embeddings/preprocessed_coles_embs/embeddings_age_coles.parquet'
embeddings_df = pd.read_parquet(embeddings_path).dropna()

embeddings_df['reg_target'] = get_regression_target(
    interval_feature=embeddings_df.post_trans_date,
    target_feature=embeddings_df.post_amount_rur,
    horizon=30 # month for AGE dataset
)

# Leave it as it was
embeddings_df['post_forecast_target'] = ([ np.sum(v == v[0])  for v in embeddings_df['post_trans_date'].values])
y = embeddings_df['post_forecast_target']
y_transformed = np.log1p(y)
embeddings_df['post_forecast_target'] =  y_transformed
embeddings_df['post_amount'] = np.array(embeddings_df['post_amount_rur'].values)
embeddings_df['post_target'] = np.array(embeddings_df['target'].values)
#post_group is ntp target
embeddings_df['post_group'] = np.array(embeddings_df['post_small_group'].values)
embeddings_df.drop(['post_small_group'], axis=1, inplace=True)
embeddings_df['post_anomaly_target'] = age_anomaly_target(embeddings_df)
# embeddings_df.to_parquet(embeddings_path_prepocessed)

In [ ]:
#plt.hist(embeddings_df['embedding'].iloc[1])

In [ ]:
transformations = {
    'original': embeddings_df['post_forecast_target'],
    'log1p': np.log1p(embeddings_df['post_forecast_target']),
    'sqrt': np.sqrt(embeddings_df['post_forecast_target']),
    'cube_root': embeddings_df['post_forecast_target'] ** (1/3),
    'fourth_root': embeddings_df['post_forecast_target'] ** 0.25,
    'log_log': np.log(np.log1p(embeddings_df['post_forecast_target'] + 1e-10)),
}

# Plot each transformation
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for (name, data), ax in zip(transformations.items(), axes.flatten()):
    ax.hist(data, bins=50, alpha=0.7)
    ax.set_title(f'{name} (skew: {stats.skew(data):.2f})')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(boxcox(embeddings_df['post_forecast_target']),bins='auto')

In [ ]:
plt.hist([ np.sum(v == v[0])  for v in embeddings_df['post_trans_date'].values])

In [ ]:
plt.hist(AGE_DF['post_forecast_target'])

# taobao

In [ ]:
def get_taobao_anomaly(path):
    # предсказание поведения пользователя: кладет товар в корзину, но не покупает его
    embed_taobao = pl.scan_parquet(path)
    coll_embed_taobao = embed_taobao.collect()
    df_seq = coll_embed_taobao.to_pandas().copy().dropna()
    # какие колонки хотим развернуть в обычный вид
    cols_to_explode = ["time", "behavior_type", "item_id"]

    # разворачиваем построчно
    pd_df_concat = (
        df_seq[["user_id"] + cols_to_explode]
        .explode(cols_to_explode)
        .reset_index(drop=True)
    )

    # приводим время и сортируем
    pd_df_concat["time"] = pd.to_datetime(pd_df_concat["time"])
    pd_df_concat = pd_df_concat.sort_values(["user_id", "time"]).reset_index(drop=True)

    ITEM_COL = "item_id"

    # 4 — добавление в корзину
    df_add_item = (
        pd_df_concat[pd_df_concat["behavior_type"] == 4]
        [["user_id", ITEM_COL, "time"]]
        .rename(columns={"time": "timestamp_add"})
    )

    # 3 — покупка
    df_buy_item = (
        pd_df_concat[pd_df_concat["behavior_type"] == 3]
        [["user_id", ITEM_COL, "time"]]
        .rename(columns={"time": "timestamp_buy"})
    )

    # для каждой пары (user, item) смотрим, был ли add без buy
    merged_item = df_add_item.merge(
        df_buy_item,
        on=["user_id", ITEM_COL],
        how="left"
    )

    # аномалия: добавил в корзину, но не купил
    anom_item = merged_item[merged_item["timestamp_buy"].isna()]

    print("Аномальных (client, item):", len(anom_item))

    # пары (user, item), где есть такая аномалия
    anom_pairs = anom_item[["user_id", ITEM_COL]].drop_duplicates()
    anom_pairs["anomaly_item_cart_no_buy"] = True

    # добавляем флаг
    pd_df_concat = pd_df_concat.merge(
        anom_pairs,
        on=["user_id", ITEM_COL],
        how="left"
    )

    pd_df_concat["anomaly_item_cart_no_buy"] = (
        pd_df_concat["anomaly_item_cart_no_buy"]
        .fillna(False)
    )

    # момент первой аномальной корзины по пользователю
    first_cart_time_item = (
        anom_item
        .groupby("user_id")["timestamp_add"]
        .min()
        .rename("first_anomaly_time_item")
    )

    pd_df_concat = pd_df_concat.merge(first_cart_time_item, on="user_id", how="left")

    # фаза относительно первой аномалии
    pd_df_concat["trget_anomaly_cart_no_buy"] = np.where(
        pd_df_concat["first_anomaly_time_item"].notna() &
        (pd_df_concat["time"] >= pd_df_concat["first_anomaly_time_item"]),
        "post",
        "pre"
    )

    pd_df_concat = pd_df_concat.sort_values(["user_id", "time"]).reset_index(drop=True)

    # возвращаем в изначальный вид
    phase_seq = (
        pd_df_concat
        .groupby("user_id")["trget_anomaly_cart_no_buy"]
        .apply(list)                      
        .reset_index()
    )

    phase_seq = phase_seq.rename(
        columns={"trget_anomaly_cart_no_buy": "trget_anomaly_cart_no_buy_seq"}
    )
    df_seq = df_seq.merge(phase_seq, on="user_id", how="left")
    post_anomaly_target = [ int('post' in v) for v in df_seq['trget_anomaly_cart_no_buy_seq'].values]
    return post_anomaly_target

In [ ]:
def check_anomaly(row):

    behavior_type = row["post_behavior_type"]
    item_id = row["post_item_id"]

    add_cart_count = {}
    purchase = set()

    for type, item in zip(behavior_type, item_id):
        if type == 3:
            add_cart_count[item] = add_cart_count.get(item, 0) + 1

        elif type == 4:
            purchase.add(item)

    is_anoamly = 0

    for item, count in add_cart_count.items():
        if count > 1 and item not in purchase:
            is_anoamly =1
            break

    return is_anoamly

In [ ]:
def get_taobao_forecast(path):
    embed_taobao = pl.scan_parquet(path)
    coll_embed_taobao = embed_taobao.collect()
    df_fore = coll_embed_taobao.to_pandas().copy().dropna()
    cols_to_explode = ['time', "item_id", "behavior_type"]
    df_fore_exp = (
        df_fore[["user_id"] + cols_to_explode]
        .explode(cols_to_explode)
        .reset_index(drop=True)
    )
    df_fore_exp["time"] = pd.to_datetime(df_fore_exp["time"])
    df_fore_exp = df_fore_exp.sort_values(["user_id", "time"]).reset_index(drop=True)
    is_transaction = df_fore_exp["behavior_type"] == 3
    df_fore_exp["last_trx_time"] = np.where(is_transaction, df_fore_exp["time"], pd.NaT)
    df_fore_exp["last_trx_time"] = (
        df_fore_exp
        .groupby("user_id")["last_trx_time"]
        .ffill()
    )
    df_fore_exp["last_trx_time"] = pd.to_datetime(df_fore_exp["last_trx_time"])
    delta = df_fore_exp["time"] - df_fore_exp["last_trx_time"]
    df_fore_exp["time_since_last_tx_days"] = delta.dt.total_seconds() / (3600 * 24)
    target_forecast_df = df_fore_exp[
        df_fore_exp["time_since_last_tx_days"].notna()
        & (df_fore_exp["time_since_last_tx_days"] > 0)
    ]
    target_seq = (
        df_fore_exp
        .groupby("user_id")["time_since_last_tx_days"]
        .apply(list)                      
        .reset_index()
    )

    df_fore_final = (
        df_fore
        .merge(target_seq, on="user_id", how="left")
    )
    return np.stack(df_fore_final["time_since_last_tx_days"])

In [ ]:
users = pd.read_csv(EMB_PATH / '../taobao/tianchi_mobile_recommend_train_user.csv')
item_to_category = {
    str(k): str(v) for k, v in zip(users['item_id'], users['item_category'])
}

In [ ]:
def remove_consecutive_duplicates(lst):
    """Remove consecutive duplicates from a list"""
    if not lst:
        return []
    result = [lst[0]]
    for i in range(1, len(lst)):
        if lst[i] != lst[i-1]:
            result.append(lst[i])
    return result

In [ ]:
def map_without_consecutive_duplicates(item_list):
    categories = []
    last_category = None
    for item_id in item_list:
        category = item_to_category.get(item_id)
        if category is not None and category != last_category:
            categories.append(category)
            last_category = category
    return categories

In [ ]:
def taobao_reg_target(data_time, unit="h", horizon=300):
    """
    log(1 + number of events in the first `horizon` time units after first post event)
    unit: "h" (hours), "D" (days), "m" (minutes), "s" (seconds)
    """
    return np.array([
        np.log1p(np.sum(((t - t[0]) / np.timedelta64(1, unit)) < horizon)) if len(t) else 0.0
        for t in data_time
    ])


In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_taobao_coles_ntp_TILL_best.parquet'

parquet_df = pd.read_parquet(embeddings_path).dropna()#.head(1000)
embeddings_df = parquet_df.copy()
embeddings_df['post_target'] = list(embeddings_df['target'].values)
# embeddings_df['post_amount'] = embeddings_df['post_behavior_type'] #[np.zeros_like(t).astype('int') for t in embeddings_df['post_time'].values]
embeddings_df["reg_target"] = taobao_reg_target(embeddings_df["post_time"], unit="h", horizon=300)
#embeddings_df['post_anomaly_target']  = get_taobao_anomaly(embeddings_path)
embeddings_df['post_anomaly_target'] = [check_anomaly(r[1]) for r in list(embeddings_df.head(None).iterrows())]
embeddings_df['post_trans_date'] = np.array(embeddings_df['post_time'].values)
embeddings_df['client_id'] = np.array(embeddings_df['user_id'].values)

post_forecast_target = get_taobao_forecast(embeddings_path)
embeddings_df['post_forecast_target'] = [np.nanmedian(v) for v in post_forecast_target]

# Map to categories without duplicates
embeddings_df['post_group'] = embeddings_df['post_item_id'].apply(map_without_consecutive_duplicates)

# Use raw item IDs instead of categories
#embeddings_df['post_group'] = embeddings_df['post_item_id'].apply(lambda lst: [key for key, _ in groupby(lst)])

#embeddings_df = embeddings_df[embeddings_df['post_group'].apply(len) > 0].copy()
#embeddings_df = embeddings_df.dropna()
# embeddings_path_prepocessed = EMB_PATH / "preprocessed_coles_embs" / "embeddings_taobao_coles.parquet"
# embeddings_df.to_parquet(embeddings_path_prepocessed)

# Rossman

In [ ]:
def rossman_reg_target(sales: pd.Series, horizon=30):
    future_sum = sales.apply(lambda x: np.sum(x[:horizon]))

    mu = future_sum.mean()
    sigma = future_sum.std()

    return (future_sum - mu) / sigma

In [ ]:
# Rossman anomaly targets (same style as view_embeddings)
def _rossman_metrics(df, horizon=60, eps=1e-6):
    metrics = {}
    metrics['sales_std'] = df['post_Sales'].apply(lambda x: np.std(np.asarray(x)[:horizon]))
    metrics['sales_sum'] = df['post_Sales'].apply(lambda x: np.sum(np.asarray(x)[:horizon]))
    metrics['customers_std'] = df['post_Customers'].apply(lambda x: np.std(np.asarray(x)[:horizon]))
    metrics['spike_ratio'] = df['post_Sales'].apply(
        lambda x: (np.max(np.asarray(x)[:horizon]) / (np.median(np.asarray(x)[:horizon]) + eps))
    )
    metrics['sales_per_customer'] = [
        (np.sum(np.asarray(s)[:horizon]) / (np.sum(np.asarray(c)[:horizon]) + eps))
        for s, c in zip(df['post_Sales'], df['post_Customers'])
    ]
    return pd.DataFrame(metrics)

def anomaly_high_sales_volatility(df, horizon=60, q=0.95):
    m = _rossman_metrics(df, horizon=horizon)
    thr = m['sales_std'].quantile(q)
    return (m['sales_std'] > thr).astype(int)

def anomaly_high_customer_volatility(df, horizon=60, q=0.95):
    m = _rossman_metrics(df, horizon=horizon)
    thr = m['customers_std'].quantile(q)
    return (m['customers_std'] > thr).astype(int)

def anomaly_extreme_sales_spike(df, horizon=60, q=0.95):
    m = _rossman_metrics(df, horizon=horizon)
    thr = m['spike_ratio'].quantile(q)
    return (m['spike_ratio'] > thr).astype(int)

def anomaly_low_sales_per_customer(df, horizon=30, q=0.05):
    m = _rossman_metrics(df, horizon=horizon)
    thr = m['sales_per_customer'].quantile(q)
    return (m['sales_per_customer'] < thr).astype(int)

def anomaly_high_sales_volume(df, horizon=60, q=0.98):
    m = _rossman_metrics(df, horizon=horizon)
    thr = m['sales_sum'].quantile(q)
    return (m['sales_sum'] > thr).astype(int)

# Usage examples (same style)
# embeddings_df['post_anomaly_target'] = anomaly_high_sales_volatility(embeddings_df)
# embeddings_df['post_anomaly_target'] = anomaly_high_customer_volatility(embeddings_df)
# embeddings_df['post_anomaly_target'] = anomaly_extreme_sales_spike(embeddings_df)
# embeddings_df['post_anomaly_target'] = anomaly_low_sales_per_customer(embeddings_df)
# embeddings_df['post_anomaly_target'] = anomaly_high_sales_volume(embeddings_df)


In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_rossman_coles_ntp_TILL_best_lied.parquet'

parquet_df = pd.read_parquet(embeddings_path).dropna()
embeddings_df = parquet_df.copy()
embeddings_df['post_trans_date'] = np.array(embeddings_df['post_Date'].values)
embeddings_df['client_id'] = np.array(embeddings_df['Store'].values)
store_info_df = pd.read_csv(EMB_PATH / '../rossman' / 'store.csv')
store_type_map = dict(zip(store_info_df['Store'], store_info_df['StoreType']))
embeddings_df['store_type_letter'] = embeddings_df['client_id'].map(store_type_map)
# Convert to integer codes
embeddings_df['post_target'] = pd.factorize(embeddings_df['store_type_letter'])[0]
embeddings_df['post_forecast_target'] = np.log1p([np.median(v) for v in embeddings_df['post_Sales']])
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: (np.sum(np.concatenate([x.Sales, x.post_Sales])) > 0.85 * 1e7).astype(int), axis=1)
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: (np.sum(x.post_Sales) > 0.6 * 1e7).astype(int), axis=1)
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: (np.sum(x.post_Sales[:30]) < 0.01 * 1e7).astype(int), axis=1)
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: np.sum(x.post_Sales[:30]) / np.sum(x.post_Customers[:30]) < 6.5, axis=1)
# embeddings_df['post_anomaly_target'] = embeddings_df.post_Customers.apply(lambda x: (np.std(x[:30]) > 4.5 * 1e3).astype(int))
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: ((np.std(x.post_Customers[:30]) / np.std(x.post_Sales[:30])) > 0.14).astype(int), axis=1)
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: (x.post_Sales[:60].sum() > 0.75 * 1e6).astype(int), axis=1)
# embeddings_df['post_anomaly_target'] = anomaly_high_sales_volatility(embeddings_df)
# embeddings_df['post_anomaly_target'] = anomaly_high_customer_volatility(embeddings_df)
embeddings_df['post_anomaly_target'] = anomaly_extreme_sales_spike(embeddings_df) ##### YAAAAAAPA It WORKS
# embeddings_df['post_anomaly_target'] = anomaly_low_sales_per_customer(embeddings_df)
# embeddings_df['post_anomaly_target'] = anomaly_high_sales_volume(embeddings_df)
# embeddings_df['post_anomaly_target'] = embeddings_df.apply(lambda x: (x.post_Sales[:60].std() > 6500).astype(int), axis=1)
embeddings_df['post_amount'] = rossman_reg_target(embeddings_df['Sales'], horizon=30)
embeddings_path_prepocessed = EMB_PATH / "preprocessed_coles_embs" / "embeddings_rossman_coles.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed, )

In [ ]:
print_count_colon_percent(embeddings_df['post_target'])

# Favorita

In [ ]:
def post_sales_reg_target(
    post_date: pd.Series,
    df: pd.DataFrame,
    horizon: int = 30,
):
    sales_cols = [c for c in df.columns if c.startswith("post_class_") and c.endswith("_sales")]

    return np.array([
        np.log1p(
            sum(
                np.sum(np.asarray(row[c])[((d - d[0]) / np.timedelta64(1, "D")) <= horizon])
                for c in sales_cols
            )
        ) if len(d) else 0.0
        for d, row in zip(post_date, df[sales_cols].to_dict("records"))
    ])

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_favorita_coles.parquet'
parquet_df = pd.read_parquet(embeddings_path)
embeddings_df = parquet_df.copy()

stores_df = pd.read_csv('../../../data/favorita/stores.csv')
embeddings_df = embeddings_df.merge(
    stores_df,
    on="store_nbr",
    how="left"
)
embeddings_df['post_trans_date'] = embeddings_df['post_date']
embeddings_df['client_id'] = embeddings_df['store_nbr']
embeddings_df['post_amount'] = post_sales_reg_target(embeddings_df.post_date, embeddings_df, horizon=30)

# Convert to integer codes
embeddings_df['post_target'] = pd.Categorical(embeddings_df['type']).codes

sales_cols = [c for c in embeddings_df.columns if c.startswith("post_class_") and c.endswith("_sales")]
embeddings_df['post_forecast_target'] = (
    embeddings_df[sales_cols]
    .apply(lambda col: col.map(lambda x: x[0] if len(x) else 0.0))
    .sum(axis=1)
)

anomaly_cities = stores_df.city.value_counts()[stores_df.city.value_counts() == 1].index.tolist()
embeddings_df['post_anomaly_target'] = embeddings_df['city'].apply(lambda x: x in anomaly_cities).astype(int)
embeddings_path_prepocessed = EMB_PATH / 'preprocessed_coles_embs' / "embeddings_favorita_coles.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed)

In [ ]:
embeddings_df['post_target'].value_counts(normalize=True).mul(100).round(3)

In [ ]:
def print_count_colon_percent(
    series,
    top_k=None,
    precision=2,
    dropna=True
):
    """
    Формат вывода:
    count : XX.XX%
    """
    counts = series.value_counts(dropna=dropna)
    percents = series.value_counts(dropna=dropna, normalize=True) * 100

    if top_k is not None:
        counts = counts.head(top_k)
        percents = percents.head(top_k)

    for value in counts.index:
        print(f"{counts[value]} : {percents[value]:.{precision}f}%")

In [ ]:
print_count_colon_percent(embeddings_df['post_target'])

# Twitter (in progress)

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_twitter'
embeddings_df = pd.read_parquet(embeddings_path).dropna()
embeddings_df['post_trans_date'] = embeddings_df['post_char_number']
embeddings_df['client_id'] = embeddings_df['tweet_id']
# Convert to integer codes
embeddings_df['post_target'] = embeddings_df['sentiment']
embeddings_df['post_group'] = embeddings_df['post_char_number']
embeddings_df['post_amount'] = embeddings_df['likes']
embeddings_path_prepocessed = EMB_PATH / 'preprocessed_coles_embs' / "embeddings_twitter.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed)

In [ ]:
embeddings_df[embeddings_df['tweet_id'] == 832].retweets

# Electric Devices

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_electric_devices_coles_ntp_TILL_best.parquet'
name = 'electric_devices'
embeddings_df = pd.read_parquet(embeddings_path).dropna()
new_column_names = {
    "sequence_id":"client_id",
    "post_time":"post_trans_date",
    "target":"post_target",
}
result = np.stack(embeddings_df['post_sequence'].values) - np.stack([v[-1] for v in embeddings_df['sequence'].values])[:, None]
first_nonzero_values = np.where(result.any(axis=1), result[np.arange(result.shape[0]), (result != 0).argmax(axis=1)], 0)
embeddings_df['post_forecast_target'] = first_nonzero_values
embeddings_df.rename(columns=new_column_names, inplace=True)
embeddings_df['post_amount'] = [np.zeros_like(t).astype('int') for t in embeddings_df['post_trans_date'].values]
# embeddings_path_prepocessed = f"/mnt/local/data/ksozykin/src/ESdB/embeddings_{name}_coles.parquet"
# embeddings_df.reset_index(drop=True).to_parquet(embeddings_path_prepocessed, index=False)

In [ ]:
print_count_colon_percent(embeddings_df['post_target'])

In [ ]:
embeddings_df['post_target'].value_counts()

# ETT

In [ ]:
method = "coles"
name = 'ett'
embeddings_path = EMB_PATH / 'coles_embs' / f'embeddings_ett_{method}_ntp_TILL_best.parquet'
embeddings_df = pd.read_parquet(embeddings_path).dropna()

new_column_names = {"week_id":"client_id", "post_time":"post_trans_date"}
forecast_target_columns = 'post_OT'

embeddings_df.rename(columns=new_column_names, inplace=True)

embeddings_df['post_amount'] = embeddings_df[forecast_target_columns]
embeddings_df['post_forecast_target'] = np.log1p([np.nanmedian(v) for v in embeddings_df[forecast_target_columns].values])
# embeddings_path_prepocessed = f"/mnt/local/data/ksozykin/src/ESdB/embeddings_{name}_{method}_{forecast_target_columns}.parquet"
# embeddings_df.reset_index(drop=True).to_parquet(embeddings_path_prepocessed, index=False)

In [ ]:
embeddings_df.iloc[20].post_date.shape

In [ ]:
embeddings_df.post_forecast_target

# 30 music

In [ ]:
new_column_names = {"user_id":"client_id", "post_timestamp":"post_trans_date", "target_play_duration_next_day":"post_amount", "post_item_id":"post_group"}

embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_30music_coles_ntp_TILL_best.parquet'
embeddings_df = pd.read_parquet(embeddings_path).dropna()
# embeddings_df.rename(columns=new_column_names, inplace=True)
# something wrong with: 
# post_trans_date pandas._libs.tslibs.np_datetime.OutOfBoundsTimedelta: Cannot cast 1391293679 from D to 'ns' without overflow.
# amount is not sequence, currently we can't use argregated targets
# embeddings_df['post_amount'] = [np.zeros_like(t).astype('int') for t in embeddings_df['post_trans_date'].values]
# embeddings_df['post_trans_date'] = [np.zeros_like(t).astype('int') for t in embeddings_df['post_trans_date'].values]

# embeddings_path_prepocessed = f"/mnt/local/data/ksozykin/src/ESdB/embeddings_{name}_coles.parquet"
# embeddings_df.reset_index(drop=True).to_parquet(embeddings_path_prepocessed, index=False)

In [ ]:
embeddings_df.columns

In [ ]:
embeddings_df.post_timestamp.apply(lambda x: (((x - x[0]) / 86400)) < 30)

In [ ]:
embeddings_df.post_item_id

In [ ]:
embeddings_df.apply(lambda x: (x.post_playtime[(((x.post_timestamp - x.post_timestamp[0]) / 86400)) < 30]).sum() <= 0, axis=1).value_counts()

In [ ]:
embeddings_df.apply(lambda x: np.log1p((x.post_playtime[(((x.post_timestamp - x.post_timestamp[0]) / 86400)) < 30]).sum()), axis=1).hist()

In [ ]:
mask = embeddings_df.apply(lambda x: (x.post_playtime[(((x.post_timestamp - x.post_timestamp[0]) / 86400)) < 30]).sum() <= 0, axis=1)

In [ ]:
embeddings_df[mask].playtime.apply(lambda x: (x == -1).sum())

In [ ]:
embeddings_df.post_item_id.apply(len).describe()

# Yambda

In [ ]:
def get_forecast_target(row):

    delta_ticks = ((row['post_timestamp'][0] - row['timestamp'][-1]) * 5)
    
    return delta_ticks
from collections import Counter

In [ ]:
def create_dislike_anomaly_target(df):

    def dislikes_ratio_func(row):

        if len(row) == 0:
            return 0.0
            
        n_dislike = (row == 3).sum()
        n_listens = (row == 1).sum()

        if n_listens == 0:
            return 0.0

        ratio = n_dislike / n_listens
        return ratio

    df["dislikes_ratio"] = df["post_event_type"].apply(dislikes_ratio_func)

    q95 = df["dislikes_ratio"].quantile(0.95)

    df["target_anomaly"] = (df["dislikes_ratio"] > q95).astype(int)

    return df["target_anomaly"]


In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_yambda_coles_ntp_TILL_best.parquet'
embeddings_df = pd.read_parquet(embeddings_path).dropna()
# new_column_names = {
#     "uid":"client_id",
#     "post_timestamp":"post_trans_date",
#     "post_track_length_seconds":"post_amount",
#     "post_is_organic":"post_target",
#     "post_item_id":"post_group"
# }
# embeddings_df['post_anomaly_target'] = create_dislike_anomaly_target(embeddings_df)
# embeddings_df['post_forecast_target'] = np.log1p(embeddings_df.apply(get_forecast_target, axis=1))
# embeddings_df.rename(columns=new_column_names, inplace=True)
# embeddings_df['post_trans_date'] = [np.zeros_like(v).astype('int') for v in embeddings_df['post_group']]
# embeddings_df['post_target'] = [Counter(v).most_common(1)[0][0] for v in embeddings_df['post_target'].values] 
# embeddings_path_prepocessed = "/mnt/local/data/ksozykin/src/ESdB/embeddings_yambda_coles.parquet"
# embeddings_df.reset_index(drop=True).to_parquet(embeddings_path_prepocessed, index=False)

In [ ]:
embeddings_df.post_track_length_seconds

In [ ]:
embeddings_df.post_timestamp

In [ ]:
embeddings_df.is_organic.apply(lambda x: np.unique(x).shape[0]).value_counts()

# Zvuk

In [ ]:
def get_zvuk_post_target(embeddings_path, artists,  max_users=150000):
    df_pd = pd.read_parquet(embeddings_path).dropna().head(max_users)
    # Load artists data
    track_to_cluster = dict(zip(artists['track_id'].astype(str), artists['cluster_id'].astype(str)))
    # EXPLODE the track_id lists first
    df_exploded = df_pd.explode('track_id').reset_index(drop=True)
    # Now map track_id to cluster_id
    df_exploded['cluster_id'] = df_exploded['track_id'].astype(str).map(track_to_cluster)
    # Filter out rows with no cluster_id
    df_exploded = df_exploded[df_exploded['cluster_id'].notna()].copy()
    # Count unique users per cluster
    unique_users = (
        df_exploded.groupby("cluster_id")["user_id"]
        .nunique()
        .rename("n_users")
    )
    # Create quartiles (handle duplicates)
    qlabels = [0, 1, 2, 3]
    cluster_meta = pd.DataFrame({
        "n_users": unique_users,
        "target_user_quartile": pd.qcut(unique_users, q=4, labels=qlabels, duplicates='drop')
    }).reset_index()
    # Merge with original data
    merged = df_exploded.merge(cluster_meta, on="cluster_id", how="left")
    # Group back into lists per user
    cols_to_list = ["datetime", "track_id", "cluster_id", "n_users", "target_user_quartile"]
    df_list = merged.groupby("user_id")[cols_to_list].agg(list).reset_index()
    return df_list["target_user_quartile"].values


def get_zvuk_post_amount(embeddings_path, max_users=150000):
    df = pd.read_parquet(embeddings_path).dropna().head(max_users)
    # разворачиваем списки в строки
    df = df.explode(["datetime", "play_duration", "track_id"]).reset_index(drop=True)
    df["date"] = pd.to_datetime(df["datetime"]).dt.date
    # получаем активность посуточную
    daily = (
        df.groupby(["user_id", "date"])
        .agg(
            total_play_duration=("play_duration", "sum")
        )
        .reset_index()
    )
    daily = daily.sort_values(["user_id", "date"])
    # создаём таргет: play_duration следующего дня
    daily["target_play_duration_next_day"] = (
        daily.groupby("user_id")["total_play_duration"].shift(-1)
    )
    # убираем последние дни (где таргета нет)
    df_targets = daily.dropna(subset=["target_play_duration_next_day"]).copy()
    user_targets = df_targets[["user_id", "target_play_duration_next_day"]].drop_duplicates()
    merged = df.merge(user_targets, on="user_id", how="left")
    # снова собираем в списки
    cols_to_list = ["datetime", "play_duration", "track_id", "target_play_duration_next_day"]
    df_list = (
        merged
        .groupby("user_id")[cols_to_list]
        .agg(list)
        .reset_index()
    )
    return df_list['target_play_duration_next_day'].values


def get_zvuk_forecast(embeddings_path, max_users=150000):
    """
    Predict user activity for next day (how many tracks they'll listen to)
    Returns: list of lists with target_activity_next_day for each user
    """
    df = pd.read_parquet(embeddings_path).dropna().head(max_users)
    # Explode lists
    df_exploded = df.explode(["datetime", "track_id"]).reset_index(drop=True)
    df_exploded["date"] = pd.to_datetime(df_exploded["datetime"]).dt.date
    # Calculate daily activity (count tracks per day)
    daily = (
        df_exploded.groupby(["user_id", "date"])
        .agg(activity=("track_id", "count"))
        .reset_index()
    )
    daily = daily.sort_values(["user_id", "date"])
    # Create target: next day's activity
    daily["target_activity_next_day"] = daily.groupby("user_id")["activity"].shift(-1)
    # Remove last day for each user (no target)
    df_targets = daily.dropna(subset=["target_activity_next_day"]).copy()
    # Get user targets
    user_targets = df_targets[["user_id", "target_activity_next_day"]].drop_duplicates()
    # Merge with original exploded data
    merged = df_exploded.merge(user_targets, on="user_id", how="left")
    # Group back into lists
    cols_to_list = ["datetime", "track_id", "target_activity_next_day"]
    df_list = merged.groupby("user_id")[cols_to_list].agg(list).reset_index()
    return df_list["target_activity_next_day"].values


def get_zvuk_anomaly(embeddings_path, artists, max_users=150000):
    """
    Detect anomalous users with high genre diversity
    Returns: list of lists with binary anomaly labels for each user
    """
    df = pd.read_parquet(embeddings_path).dropna().head(max_users)
    # Load artists data and create track->cluster mapping
    track_to_cluster = dict(zip(artists['track_id'].astype(str), artists['cluster_id'].astype(str)))
    # Explode track_id lists
    df_exploded = df.explode(["datetime", "track_id"]).reset_index(drop=True)
    # Map track_id to cluster_id
    df_exploded['cluster_id'] = df_exploded['track_id'].astype(str).map(track_to_cluster)
    # Filter out rows with no cluster_id
    df_exploded = df_exploded[df_exploded['cluster_id'].notna()].copy()
    # Calculate user diversity statistics
    total_events = df_exploded.groupby("user_id")["cluster_id"].count().rename("total_clusters")
    unique_clusters = df_exploded.groupby("user_id")["cluster_id"].nunique().rename("unique_clusters")
    # Create ratio
    cluster_profile = unique_clusters.to_frame().join(total_events)
    cluster_profile["ratio"] = cluster_profile["unique_clusters"] / cluster_profile["total_clusters"]
    # Find 95th percentile threshold
    q95 = cluster_profile["ratio"].quantile(0.95)
    # Mark anomalies
    cluster_profile["target_is_high_diversity"] = (cluster_profile["ratio"] >= q95).astype(int)
    # Merge back with original data
    merged = df_exploded.merge(
        cluster_profile[["target_is_high_diversity"]].reset_index(),
        on="user_id",
        how="left"
    )
    # Group back into lists
    cols_to_list = ["datetime", "track_id", "cluster_id", "target_is_high_diversity"]
    df_list = merged.groupby("user_id")[cols_to_list].agg(list).reset_index()
    return df_list["target_is_high_diversity"].values

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_zvuk_coles_ntp_TILL_best.parquet'
max_users = 30000
embeddings_df = pd.read_parquet(embeddings_path).dropna().head(max_users)

# artists = pd.read_parquet('/mnt/local/data/ksozykin/src/ESdB/raw_embd/zvuk-track_artist_embedding.parquet')
# t1 = get_zvuk_post_target(embeddings_path, artists, max_users=max_users)
# t2 = get_zvuk_post_amount(embeddings_path, max_users=max_users)
# t3 = get_zvuk_forecast(embeddings_path, max_users=max_users)
# t4 = get_zvuk_anomaly(embeddings_path, artists, max_users=max_users)


# embeddings_df['post_trans_date'] = np.array(embeddings_df['post_datetime'].values)
# embeddings_df['client_id'] = np.array(embeddings_df['user_id'].values)
# embeddings_df['post_target'] = np.array([ Counter(v).most_common(1)[0][0] for v in t1])
# embeddings_df['post_amount'] = [v for v in t2]
# embeddings_df['post_forecast_target'] = [v for v in t3]
# embeddings_df['post_forecast_target'] = np.log1p([np.nanmedian(v) for v in embeddings_df['post_forecast_target'].values])
# embeddings_df['post_anomaly_target'] = [np.any(v) for v in t4]
# embeddings_df['post_group'] = embeddings_df['post_track_id'].apply(lambda lst: [key for key, _ in groupby(lst)])
# embeddings_path_prepocessed = f"/mnt/local/data/ksozykin/src/ESdB/embeddings_{name}_{method}.parquet"
# embeddings_df.to_parquet(embeddings_path_prepocessed)

In [ ]:
embeddings_df.columns

In [ ]:
embeddings_df.post_play_duration.apply(lambda x: x[0]).describe()